In [1]:
"""
Summarize fuel treatment acres within fire perimeters (TWIG interactions).
Maxwell.Cook@colostate.edu

Treatments are classified with the multi-attribute crosswalk from the
`treatment_interactions` package (activity / type / method / twig_category), replacing
the old single `type`-column allow-list that silently dropped null-`type` records.
Combined thin / burn / removal variables are built as a mutually-exclusive (disjoint)
area split per fire so predictors do not reuse acres (avoids collinearity). A change
report vs the previous (V8) variables is written to data/tabular/qa/.
No CFT integration here -- TWIG side only.
"""
import os, sys
from pathlib import Path

# treatment_interactions: TWIG harmonization to the CFT ACTIVITY vocabulary.
# Editable-installed; fall back to the source tree on sys.path if not installed.
try:
    import treatment_interactions as ci
except ModuleNotFoundError:
    CI_CODE = Path("/Users/mcc/Library/CloudStorage/Box-Box/MCC/projects/treatment_interactions/code")
    sys.path.insert(0, str(CI_CODE))
    import treatment_interactions as ci
from treatment_interactions import clean_geometry, harmonize_twig_activity

# custom functions (imports gpd, pd, np, tqdm, ... and combined_treatment_vars)
sys.path.append(os.getcwd())
from __functions import *

datadir = '/Users/mcc/Library/CloudStorage/Box-Box/MCC/data/'
projdir = os.path.dirname(os.getcwd())
qadir   = os.path.join(projdir, 'data/tabular/qa')
os.makedirs(qadir, exist_ok=True)
print(projdir)
print("Ready !")

/Users/mcc/Library/CloudStorage/Box-Box/MCC/projects/ReSHAPE/valuation
Ready !


In [2]:
# load the incidents / perimeters
incis = gpd.read_file(os.path.join(projdir, 'data/spatial/mod/ics209plus_2014to2023_perimeters_tcc.gpkg'))
incis = incis[['INCIDENT_ID','INCIDENT_NAME','DISCOVERY_DATE','FUEL_MODEL','FINAL_ACRES','geometry']]
print(f"Processing for {len(incis)} fires.")
print(f"CRS: {incis.crs}")

Processing for 6940 fires.
CRS: EPSG:5070


In [19]:
incis[incis['INCIDENT_ID'] == '2019_10665633_SWAN LAKE'].head()

,INCIDENT_ID,INCIDENT_NAME,DISCOVERY_DATE,FUEL_MODEL,FINAL_ACRES,geometry
29,2019_10665633_SWAN LAKE,SWAN LAKE,2019-06-05 21:52:00,Timber (Grass and Understory),167182.9,"MULTIPOLYGON (((-3128452.243 5034651.4, -31284..."


In [3]:
# load the fuel treatment data (within 1 km of fires; produced by 02a)
fp = os.path.join(projdir, 'data/spatial/mod/treatments/twig_treatments_1km.gpkg')
twig = gpd.read_file(fp)
# convert date fields and extract the completion year
twig['treatment_date'] = pd.to_datetime(twig['treatment_date'], unit='ms', errors='coerce')
twig['actual_completion_date'] = pd.to_datetime(twig['actual_completion_date'], unit='ms', errors='coerce')
twig['year_comp'] = twig['actual_completion_date'].dt.year
print(f"Loaded {len(twig):,} TWIG records; CRS: {twig.crs}")
twig[['INCIDENT_ID','treatment_date','year_comp','type','activity','method','twig_category','acres']].head()

Loaded 76,419 TWIG records; CRS: EPSG:3857


,INCIDENT_ID,treatment_date,year_comp,type,activity,method,twig_category,acres
0,2019_10665633_SWAN LAKE,2018-09-16,2018.0,Thinning,None,None,Mechanical,4.713520
1,2019_10665633_SWAN LAKE,2019-03-04,2019.0,Mastication,None,None,Mechanical,21.531993
2,2019_10665633_SWAN LAKE,2018-11-19,2018.0,Hand Pile Burn,None,None,Planned Ignition,1.173396
3,2019_10665633_SWAN LAKE,2018-09-24,2018.0,Hand Pile Burn,None,None,Planned Ignition,30.485169
4,2019_10665633_SWAN LAKE,2018-09-16,2018.0,Thinning,None,None,Mechanical,1.401910


In [4]:
# clean the TWIG geometry
print(f"Before cleaning: {len(twig):,}")
twig = clean_geometry(twig)
print(f"After cleaning:  {len(twig):,}")

Before cleaning: 76,419
After cleaning:  76,402


## Classify treatments (multi-attribute harmonization)

Each record is resolved to a single CFT `ACTIVITY` across `activity` -> `type` ->
`method` -> `twig_category` (canopy-wins, then field specificity, then confidence).
Generic thins are split into `Manual` / `Mechanical` by equipment/method. True
non-fuels (chemical, grazing, seeding, survey) are **dropped**; anything the crosswalk
cannot resolve is **quarantined** (`ACTIVITY=None`, logged) rather than silently
dropped. This replaces the old single-`type` allow-list that dropped null-`type`
records carrying a valid `activity`.

In [5]:
# harmonize -> ACTIVITY (+ provenance columns), and an audit report
twig_h, harmonize_report = harmonize_twig_activity(twig, acres_col="acres")

# keep only resolved fuels treatments
trts = twig_h[twig_h["ACTIVITY"].notna()].copy()

# save the audit (resolved / dropped / quarantined + descriptor fields) for review
harmonize_report.to_csv(os.path.join(qadir, 'twig_harmonize_report.csv'), index=False)
print(trts['ACTIVITY'].value_counts())

TWIG harmonization (footprint acres):
  resolved     71,512 records  36,817,649 ac
  dropped      4,881 records  37,724,280 ac
  quarantined      9 records       8,683 ac
  (classify_thin split 22,959 'Thin' records into Manual/Mechanical)

[quarantine] 9 records (8,683 ac) unresolved — extend the crosswalk:
                                                               size          sum
activity                    type method         twig_category                   
Wildfire - Natural Ignition N/A  Remote Sensing NaN               1  8147.339480
NaN                         NaN  NaN            NaN               8   535.612443
ACTIVITY
Mechanical         18964
Manual             11923
Broadcast Burn     11328
Pile Burn          11197
Pile Fuels          8671
Removal             4480
Lop and Scatter     2746
Mastication         2203
Name: count, dtype: int64


In [6]:
# anything the crosswalk could not resolve -> extend data/twig_crosswalk.csv to fix
q = harmonize_report[harmonize_report['ACTIVITY_STATUS'] == 'quarantined']
print(f"Quarantined (unresolved) records: {len(q)}")
q.head(20)

Quarantined (unresolved) records: 9


,activity,type,method,twig_category,equipment,ACRES_GEO,ACRES_REPORTED,ACTIVITY,ACTIVITY_FIELD,ACTIVITY_CONF,ACTIVITY_STATUS,RAN_CLASSIFY_THIN
1193,None,None,None,None,None,117.189917,81.347073,None,None,None,quarantined,False
1220,None,None,None,None,None,117.189917,81.347073,None,None,None,quarantined,False
31319,Wildfire - Natural Ignition,N/A,Remote Sensing,None,NA,8147.339480,4126.412000,None,None,None,quarantined,False
40019,None,None,None,None,None,8.545632,5.624252,None,None,None,quarantined,False
40020,None,None,None,None,None,7.320733,4.827183,None,None,None,quarantined,False
40453,None,None,None,None,None,216.336350,146.092405,None,None,None,quarantined,False
64749,None,None,None,None,None,23.102464,14.912193,None,None,None,quarantined,False
64808,None,None,None,None,None,7.163821,3.789484,None,None,None,quarantined,False
64824,None,None,None,None,None,38.763608,20.573143,None,None,None,quarantined,False


## Colocate & clip treatments to the 1 km fire buffer

In [7]:
# buffer perimeters by 1 km and clip each fire's treatments to that buffer
print("Buffering fire perimeters (1 km) and clipping treatments ...")
buffer = incis.copy().to_crs(trts.crs)              # match treatment CRS
buffer['geometry'] = buffer.geometry.buffer(1000)
buffer['fire_buffer_acres'] = buffer.geometry.area / 4046.86

trt_clips = []
for _, fire_row in tqdm(buffer.iterrows(), total=len(buffer), desc='Processing Fires'):
    sub = trts[trts['INCIDENT_ID'] == fire_row['INCIDENT_ID']]
    if sub.empty:
        continue
    clipped = gpd.clip(sub, fire_row['geometry'])
    if clipped.empty:
        continue
    clipped = clipped.copy()
    clipped['fire_buffer_acres'] = fire_row['fire_buffer_acres']
    trt_clips.append(clipped)

trts_fire = gpd.GeoDataFrame(pd.concat(trt_clips, ignore_index=True), crs=trts.crs)
trts_fire['gis_acres'] = trts_fire.geometry.area / 4046.86
print(f"{len(trts_fire):,} clipped treatment polygons across "
      f"{trts_fire['INCIDENT_ID'].nunique():,} fires")

Buffering fire perimeters (1 km) and clipping treatments ...


Processing Fires:   0%|          | 0/6940 [00:00<?, ?it/s]

55,723 clipped treatment polygons across 1,489 fires


## Per-type treatment shares (long format)

One row per fire x `ACTIVITY`: the dissolved footprint (removing within-type overlap),
its acreage, the percent of the 1 km buffer treated, and the number of distinct
treatment years.

In [8]:
# per-type footprint: union every polygon of each ACTIVITY within a fire
# (dissolve by fire x ACTIVITY only -- NOT by year, which would drop null-year records)
per_type = (trts_fire[['INCIDENT_ID','ACTIVITY','fire_buffer_acres','geometry']]
            .dissolve(by=['INCIDENT_ID','ACTIVITY']).reset_index())
per_type['treatment_acres'] = per_type.geometry.area / 4046.86
per_type['pct_treated'] = per_type['treatment_acres'] / per_type['fire_buffer_acres'] * 100

# treatment years (list) and distinct-year count per fire x type (null years omitted)
yrs = (trts_fire.dropna(subset=['year_comp'])
       .groupby(['INCIDENT_ID','ACTIVITY'])['year_comp']
       .agg(lambda x: sorted(set(x.astype(int)))).reset_index(name='treatment_years'))
n_trts = (trts_fire.groupby(['INCIDENT_ID','ACTIVITY'])
          .agg(n_treatments=('year_comp','nunique')).reset_index())
per_type = (per_type.merge(yrs, on=['INCIDENT_ID','ACTIVITY'], how='left')
                    .merge(n_trts, on=['INCIDENT_ID','ACTIVITY'], how='left'))

per_type_long = per_type[['INCIDENT_ID','ACTIVITY','treatment_acres','fire_buffer_acres',
                          'pct_treated','treatment_years','n_treatments','geometry']].rename(
                          columns={'ACTIVITY':'treatment_type'})
print(per_type_long['treatment_type'].value_counts())
per_type_long.head()

treatment_type
Broadcast Burn     955
Mechanical         826
Manual             622
Pile Burn          591
Pile Fuels         509
Mastication        310
Lop and Scatter    294
Removal            288
Name: count, dtype: int64


,INCIDENT_ID,treatment_type,treatment_acres,fire_buffer_acres,pct_treated,treatment_years,n_treatments,geometry
0,2014_1075636_DOG ROCK,Mechanical,33.406147,2704.683614,1.235122,[2009],1,"POLYGON ((-13329367 4536569, -13329365 4536558..."
1,2014_1137798_135A,Broadcast Burn,1220.046406,2126.445588,57.374918,"[2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013]",8,"MULTIPOLYGON (((-9459595 3498513, -9459665 349..."
2,2014_1138291_WEST RANGE FIRE,Broadcast Burn,63.143029,7025.807641,0.898730,[2014],1,"MULTIPOLYGON (((-11889298 5516755, -11889317 5..."
3,2014_1138291_WEST RANGE FIRE,Mechanical,103.670747,7025.807641,1.475571,"[2010, 2011, 2013]",3,"MULTIPOLYGON (((-11890294 5516519, -11890282 5..."
4,2014_1138291_WEST RANGE FIRE,Pile Fuels,16.442130,7025.807641,0.234025,"[2010, 2011]",2,"MULTIPOLYGON (((-11890310 5516571, -11890294 5..."


## Mutually-exclusive combined variables (thin / burn / removal)

`combined_treatment_vars` unions each fire's polygons into THIN, BURN, REMOVAL and
FUELRED (=BURN or REMOVAL) footprints and derives **disjoint** pieces so predictors do
not reuse acres:

- **Set A (thin x burn):** `thin_only`, `burn_only`, `thin_x_burn`
- **Set B (thin x fuels-reduction):** `thin_only_fr`, `fuelred_only`, `thin_x_fuelred`
- **Marginal totals (reference):** `all_thin`, `all_burn`, `all_removal`, `all_fuelred`
- **`thin_x_broadcast`** (thin ∩ broadcast-burn-only) is kept only to compare against the
  old broadcast-only `Thin + Rx Fire` in the change report.

Within a set the pieces sum to the union footprint (no double-count). Sets A and B are
alternative model specifications -- do not use both, or a marginal total together with
its own disjoint pieces, in one model. `thin_x_fuelred` >= `thin_x_burn` by construction.

In [9]:
# disjoint + marginal thin/burn/removal variables per fire (acres and percent-of-buffer)
combined = combined_treatment_vars(trts_fire, buffer)
_show = ['all_thin','all_burn','thin_only','burn_only','thin_x_burn',
         'thin_only_fr','fuelred_only','thin_x_fuelred','thin_x_broadcast']
print(combined[_show].describe().round(3).to_string())
combined.head()

       all_thin  all_burn  thin_only  burn_only  thin_x_burn  thin_only_fr  fuelred_only  thin_x_fuelred  thin_x_broadcast
count  1489.000  1489.000   1489.000   1489.000     1489.000      1489.000      1489.000        1489.000          1489.000
mean      4.104    15.397      2.054     13.347        2.050         1.918        13.501           2.186             1.261
std       8.716    25.105      5.766     23.746        5.877         5.590        23.745           6.147             4.851
min       0.000     0.000      0.000      0.000        0.000         0.000         0.000           0.000             0.000
25%       0.000     0.124      0.000      0.000        0.000         0.000         0.000           0.000             0.000
50%       0.673     3.063      0.090      1.299        0.000         0.064         1.480           0.003             0.000
75%       4.276    18.235      1.572     13.722        1.168         1.349        14.110           1.339             0.172
max      73.837 

,INCIDENT_ID,fire_buffer_acres,all_thin_acres,all_thin,all_burn_acres,all_burn,all_removal_acres,all_removal,all_fuelred_acres,all_fuelred,...,thin_x_burn_acres,thin_x_burn,thin_only_fr_acres,thin_only_fr,fuelred_only_acres,fuelred_only,thin_x_fuelred_acres,thin_x_fuelred,thin_x_broadcast_acres,thin_x_broadcast
0,2014_1075636_DOG ROCK,2704.683614,33.406147,1.235122,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,33.406147,1.235122,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,2014_1137798_135A,2126.445588,0.000000,0.000000,1220.046406,57.374918,0.000000,0.000000,1220.046406,57.374918,...,0.000000,0.000000,0.000000,0.000000,1220.046406,57.374918,0.000000,0.000000,0.000000,0.000000
2,2014_1138291_WEST RANGE FIRE,7025.807641,103.670747,1.475571,63.143029,0.898730,7.545109,0.107391,70.688138,1.006121,...,63.143029,0.898730,32.982609,0.469449,0.000000,0.000000,70.688138,1.006121,63.143029,0.898730
3,2014_247609_SODA,7293.605224,0.000000,0.000000,14.065597,0.192848,0.000000,0.000000,14.065597,0.192848,...,0.000000,0.000000,0.000000,0.000000,14.065597,0.192848,0.000000,0.000000,0.000000,0.000000
4,2014_257437_FROST BITE,7890.723274,211.322111,2.678108,2.841339,0.036009,0.000000,0.000000,2.841339,0.036009,...,2.841339,0.036009,208.480773,2.642100,0.000000,0.000000,2.841339,0.036009,2.841339,0.036009


## Assemble the wide per-fire table

In [10]:
# individual ACTIVITY shares -> wide (percent of buffer)
type_wide = (per_type_long
    .pivot_table(index='INCIDENT_ID', columns='treatment_type', values='pct_treated',
                 aggfunc='sum', fill_value=0)
    .reset_index())

# percent columns from the combined (disjoint + marginal) variables
pct_cols = ['INCIDENT_ID','fire_buffer_acres',
            'all_thin','all_burn','all_removal','all_fuelred',
            'thin_only','burn_only','thin_x_burn',
            'thin_only_fr','fuelred_only','thin_x_fuelred',
            'thin_x_broadcast']
wide = type_wide.merge(combined[pct_cols], on='INCIDENT_ID', how='outer')
print(f"{len(wide):,} fires with treatments")
wide.head()

1,489 fires with treatments


,INCIDENT_ID,Broadcast Burn,Lop and Scatter,Manual,Mastication,Mechanical,Pile Burn,Pile Fuels,Removal,fire_buffer_acres,...,all_burn,all_removal,all_fuelred,thin_only,burn_only,thin_x_burn,thin_only_fr,fuelred_only,thin_x_fuelred,thin_x_broadcast
0,2014_1075636_DOG ROCK,0.000000,0.0,0.0,0.000000,1.235122,0.000000,0.000000,0.000000,2704.683614,...,0.000000,0.000000,0.000000,1.235122,0.000000,0.000000,1.235122,0.000000,0.000000,0.000000
1,2014_1137798_135A,57.374918,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,2126.445588,...,57.374918,0.000000,57.374918,0.000000,57.374918,0.000000,0.000000,57.374918,0.000000,0.000000
2,2014_1138291_WEST RANGE FIRE,0.898730,0.0,0.0,0.000000,1.475571,0.000000,0.234025,0.107391,7025.807641,...,0.898730,0.107391,1.006121,0.576841,0.000000,0.898730,0.469449,0.000000,1.006121,0.898730
3,2014_247609_SODA,0.000000,0.0,0.0,0.096496,0.000000,0.192848,0.192865,0.000000,7293.605224,...,0.192848,0.000000,0.192848,0.000000,0.192848,0.000000,0.000000,0.192848,0.000000,0.000000
4,2014_257437_FROST BITE,0.036009,0.0,0.0,0.000000,2.678108,0.000000,0.000000,0.000000,7890.723274,...,0.036009,0.000000,0.036009,2.642100,0.000000,0.036009,2.642100,0.000000,0.036009,0.036009


## Previous wildfire burned area (within 1 km)

In [11]:
# percent of each fire's 1 km buffer that previously burned (from 02a colocation)
fp = os.path.join(projdir, 'data/spatial/mod/treatments/previous_burns_1km.gpkg')
fires = gpd.read_file(fp)
fires_flat  = fires.dissolve(by=['INCIDENT_ID']).reset_index()
fires_union = gpd.GeoDataFrame(geometry=[fires_flat.union_all()], crs=fires_flat.crs)

intersections = gpd.overlay(buffer[['INCIDENT_ID','geometry']],
                            fires_union.to_crs(buffer.crs), how='intersection')
intersections['overlap_acres'] = intersections.geometry.area / 4046.86
prev = intersections.groupby('INCIDENT_ID', as_index=False)['overlap_acres'].sum()
prev = prev.merge(buffer[['INCIDENT_ID','fire_buffer_acres']], on='INCIDENT_ID', how='left')
prev['Wildfire'] = prev['overlap_acres'] / prev['fire_buffer_acres'] * 100

wide = wide.merge(prev[['INCIDENT_ID','Wildfire']], on='INCIDENT_ID', how='left')
wide['Wildfire'] = wide['Wildfire'].fillna(0)
print(prev['Wildfire'].describe())

count    1178.000000
mean       46.202783
std        32.842080
min         0.000071
25%        14.535257
50%        44.496783
75%        72.991073
max       100.000000
Name: Wildfire, dtype: float64


In [12]:
# spot check: Cameron Peak (2020)
wide[wide['INCIDENT_ID'].str.contains('CAMERON PEAK', case=False, na=False)].T

,952
INCIDENT_ID,2020_11856938_CAMERON PEAK
Broadcast Burn,0.977067
Lop and Scatter,0.034488
Manual,0.537819
Mastication,0.015313
Mechanical,0.592673
Pile Burn,1.474737
Pile Fuels,1.051022
Removal,5.135571
fire_buffer_acres,467681.845041


In [13]:
# save long + wide summaries
per_type_long.drop(columns='geometry').to_csv(
    os.path.join(projdir,'data/tabular/MTBS_TWIG_summary_long-format.csv'), index=False)
wide.to_csv(os.path.join(projdir,'data/tabular/TWIG_summary_wide-format.csv'), index=False)
print("Saved long + wide summaries.")

Saved long + wide summaries.


## Change / sanity report vs the previous (V8) variables

Separates change from (a) the **classifier swap** -- more records retained by the
multi-attribute harmonization -- from (b) **definitional broadening** (all-burn vs the
old broadcast-only `Thin + Rx Fire`). `thin_x_broadcast` is the apples-to-apples
counterpart of the old `Thin + Rx Fire`.

In [14]:
# dataset-level recovery: old single-`type` allow-list vs new harmonization
old_keep = ['Thinning','Machine Pile','Hand Pile','Mastication','Lop and Scatter',
            'Broadcast Burn','Jackpot Burn','Fire Use','Machine Pile Burn',
            'Hand Pile Burn','Biomass Removal']
n_old = twig['type'].isin(old_keep).sum()
n_new = len(trts)
print(f"Records kept -- old type-only filter: {n_old:,} | new harmonization: {n_new:,} "
      f"(net {n_new-n_old:+,})")
recovered = twig_h[twig_h['ACTIVITY'].notna() & ~twig_h['type'].isin(old_keep)]
print(f"Recovered (resolved but NOT in old type list): {len(recovered):,}")
print("  by deciding descriptor field:")
print(recovered['ACTIVITY_FIELD'].value_counts().to_string())

Records kept -- old type-only filter: 56,735 | new harmonization: 71,512 (net +14,777)
Recovered (resolved but NOT in old type list): 15,972
  by deciding descriptor field:
ACTIVITY_FIELD
activity         11157
method            2041
type              1901
twig_category      873


In [15]:
# per-fire old-vs-new for the comparable variables
v8_fp = os.path.join(projdir, 'data/tabular/incidents/xlsx',
                     'V8 fuller data 7-17-26 2025dollars--all thin all burn only.xlsx')
old = pd.read_excel(v8_fp, sheet_name='FullMerge')

# NOTE: V8 FullMerge holds the modeling subset of fires; compare on the SHARED fires
# (inner join) so deltas reflect the workflow change, not a change in fire coverage.
old_new_map = {
    'Manual Thin':'Manual', 'Mechanical Thin':'Mechanical',
    'Broadcast Burn':'Broadcast Burn', 'Biomass Removal':'Removal',
    'Mastication':'Mastication', 'Lop and Scatter':'Lop and Scatter',
    'All thin':'all_thin',
    'Thin + Rx Fire':'thin_x_broadcast',   # old = thin ∩ broadcast-only (apples-to-apples)
}
parts = []
for oc, nc in old_new_map.items():
    if oc not in old.columns or nc not in wide.columns:
        print(f"  (skip {oc!r} -> {nc!r}: missing)"); continue
    # rename before merge so oc == nc (e.g. 'Broadcast Burn') does not collide into _x/_y
    o = old[['INCIDENT_ID', oc]].rename(columns={oc: 'old'})
    n = wide[['INCIDENT_ID', nc]].rename(columns={nc: 'new'})
    m = o.merge(n, on='INCIDENT_ID', how='inner')
    m['old'] = pd.to_numeric(m['old'], errors='coerce')
    m['new'] = pd.to_numeric(m['new'], errors='coerce')
    parts.append(pd.DataFrame({'INCIDENT_ID':m['INCIDENT_ID'],
                               'variable':f'{oc} -> {nc}','old':m['old'],'new':m['new']}))
change = pd.concat(parts, ignore_index=True)
change['delta'] = change['new'].fillna(0) - change['old'].fillna(0)
change.to_csv(os.path.join(qadir,'TWIG_change_report_perfire.csv'), index=False)

totals = (change.groupby('variable')
    .agg(old_mean=('old','mean'), new_mean=('new','mean'),
         old_sum=('old','sum'),  new_sum=('new','sum'),
         mean_delta=('delta','mean'),
         n_fires_changed=('delta', lambda s: int((s.abs() > 0.01).sum())))
    .round(3))
totals.to_csv(os.path.join(qadir,'TWIG_change_report_totals.csv'))
print(totals.to_string())
n_shared = old['INCIDENT_ID'].isin(wide['INCIDENT_ID']).sum()
print(f"\n[coverage] V8 fires: {old['INCIDENT_ID'].nunique():,} | "
      f"new fires w/ treatment: {wide['INCIDENT_ID'].nunique():,} | shared: {n_shared:,}")

                                    old_mean  new_mean   old_sum   new_sum  mean_delta  n_fires_changed
variable                                                                                               
All thin -> all_thin                   4.474     4.629  3279.500  3392.760       0.155              400
Biomass Removal -> Removal             0.902     0.830   661.318   608.225      -0.072              115
Broadcast Burn -> Broadcast Burn      10.289    11.886  7541.777  8712.442       1.597              203
Lop and Scatter -> Lop and Scatter     0.513     0.607   375.961   444.611       0.094               44
Manual Thin -> Manual                  1.209     1.894   886.474  1388.147       0.684              256
Mastication -> Mastication             0.049     0.535    35.554   391.847       0.486              127
Mechanical Thin -> Mechanical          2.362     3.234  1731.708  2370.797       0.872              315
Thin + Rx Fire -> thin_x_broadcast     0.957     1.259   701.258

## Deliver the cleaned variables (standalone, joinable on INCIDENT_ID)

In [16]:
# cleaned variables (percent of 1 km buffer), one row per incident (0 where untreated)
clean_cols = ['INCIDENT_ID',
    'Manual','Mechanical','Broadcast Burn','Pile Burn','Pile Fuels','Mastication',
    'Lop and Scatter','Removal',
    'all_thin','all_burn','all_removal','all_fuelred',
    'thin_only','burn_only','thin_x_burn',
    'thin_only_fr','fuelred_only','thin_x_fuelred',
    'thin_x_broadcast','Wildfire']
clean = wide.reindex(columns=[c for c in clean_cols if c in wide.columns])
clean = incis[['INCIDENT_ID']].merge(clean, on='INCIDENT_ID', how='left')
val_cols = [c for c in clean.columns if c != 'INCIDENT_ID']
clean[val_cols] = clean[val_cols].fillna(0)

out_fp = os.path.join(projdir,'data/tabular/TWIG_clean_vars.csv')
clean.to_csv(out_fp, index=False)
print(f"Saved cleaned variables for {len(clean):,} incidents -> {out_fp}")
clean.head()

Saved cleaned variables for 6,940 incidents -> /Users/mcc/Library/CloudStorage/Box-Box/MCC/projects/ReSHAPE/valuation/data/tabular/TWIG_clean_vars.csv


,INCIDENT_ID,Manual,Mechanical,Broadcast Burn,Pile Burn,Pile Fuels,Mastication,Lop and Scatter,Removal,all_thin,...,all_removal,all_fuelred,thin_only,burn_only,thin_x_burn,thin_only_fr,fuelred_only,thin_x_fuelred,thin_x_broadcast,Wildfire
0,2021_12722173_PILOT POINT,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2015_2896784_TWIN CREEKS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2022_14549956_CONTACT CREEK,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2015_2815113_COPENHAGEN CREEK,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2015_2716445_PAULS CREEK,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [18]:
# make a subset that includes only fires with some thin or burn
clean_thin = clean[(clean['all_thin']>0) | (clean['all_burn']>0)]
len(clean_thin)

1431

In [17]:
# optional: add the cleaned variables as a NEW sheet on a COPY of the V8 workbook
# (leaves the original FullMerge sheet untouched)
import shutil
dst = v8_fp.replace('.xlsx', ' (+TWIG_clean_v2).xlsx')
shutil.copy(v8_fp, dst)
with pd.ExcelWriter(dst, engine='openpyxl', mode='a', if_sheet_exists='replace') as xw:
    clean.to_excel(xw, sheet_name='TWIG_clean_v2', index=False)
print(f"Wrote sheet 'TWIG_clean_v2' -> {dst}")

Wrote sheet 'TWIG_clean_v2' -> /Users/mcc/Library/CloudStorage/Box-Box/MCC/projects/ReSHAPE/valuation/data/tabular/incidents/xlsx/V8 fuller data 7-17-26 2025dollars--all thin all burn only (+TWIG_clean_v2).xlsx


In [ ]:
# find matching events
clean_f = clean[clean['INCIDENT_ID'].isin()]